In [1]:
from column import TNN_Col
from layer import Layer
from submodule import *
from backend.backend import * 
import argparse
import os
from rich.console import Console
from tnn_mdls.func_mdls import *
from tnn_mdls.tb_func_mdls import *
import time
from veriloggen import *
import copy
from model import Model

In [2]:
myModel = Model()
myModel.add(Layer(layer_type="TNN", num_col=2, num_neurons=4, num_dend=2, p_dist=2, p_prox=1, num_seg=3, wres_dist=3, wres_prox=3, thres=6))

In [4]:
myModel.add(Layer(layer_type="TNN", num_col=1, num_neurons=4, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6))

In [5]:
layers = []

In [6]:
myModel.layers

In [4]:
temp = Layers(num_col=2, num_neurons=4, num_dend=2, p_dist=2, p_prox=1, num_seg=3, wres_dist=3, wres_prox=3, thres=6)
layers.append(temp)

In [4]:
temp = Layers.TNN_Layer(num_col=1, num_neurons=4, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6)
layers.append(temp)

In [5]:
temp = Layers.TNN_Layer(num_col=1, num_neurons=2, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6)
layers.append(temp)

In [6]:
layers

In [7]:
model = Module('model')
clk = model.Input('clk')
grst = model.Input('grst')
rstb = model.Input('rstb')

In [8]:
# Generate model (wrapper) ports
for i in range(len(layers)):
    layer = layers[i]
    ports = copy.deepcopy(layer.get_ports())
    params = copy.deepcopy(layer.get_params())
    
    # Copy all params to model
    for key in params:
        param = params[key]
        param_name = param.name
        param.name = 'L'+str(i)+'_'+param_name
        model.add_object(param)
        
    # model input ports should match layer 1 input ports
    if (i==0):
        # iterate through ports
        for key in ports:
            port = ports[key]
            port_name = port.name
            # skip clk, grst, rstb
            if ((port_name!='clk') & (port_name!='grst') & (port_name!='rstb')):
                # add input ports to model
                if (isinstance(port, core.vtypes.Input)):
                    if (port_name.startswith('input_spikes_dist')):
                        port.name = 'model_input'
                    else:
                        port.name = 'L0_'+port.name
                    model.add_object(port)
    # add non-connecting input ports to model for other layers
    else:
        # iterate through ports
        for key in ports:
            port = ports[key]
            port_name = port.name
            # skip clk, grst, rstb
            if ((port_name!='clk') & (port_name!='grst') & (port_name!='rstb')):
                # add input ports to model
                if (isinstance(port, core.vtypes.Input)):
                    # filter out connecting ports
                    if (not(port_name.startswith('input_spikes_dist'))):
                        port.name = 'L'+str(i)+'_'+port.name
                        model.add_object(port)
                        
        # generate output port
        if (i == len(layers)-1):
            for key in ports:
                port = ports[key]
                port_name = port.name
                if (isinstance(port, core.vtypes.Output)):
                    port.name = 'model_output'
                    model.add_object(port)

In [9]:
model_ports = model.get_ports()
model_params = model.get_params()

for i in range(len(layers)):
    layer = layers[i]
    layer_ports = [clk, grst, rstb]
    layer_params = [i]
    
    # Add params
    for key in model_params:
        if (key).startswith('L'+str(i)):
            print(key)
            layer_params.append(model_params[key])
    
    # For first layer, add all ports with prefix L0
    if i == 0:
        for key in model_ports:
            if ((key).startswith('L'+str(i)) | (key).startswith('model_input')):
                layer_ports.append(model_ports[key])
                
        # Instantiate wire to connect output to next layer's input
        neuron_count = layers[i].get_params()['NUM_NEURONS'].value * layers[i].get_params()['NUM_COL'].value
        last_out = model.Wire('out_'+str(i)+'_in_'+str(i+1), neuron_count)
        layer_ports.append(last_out)
    else:
        # distal input is last layer's output
        layer_ports.append(last_out)
        
        for key in model_ports:
            if (key).startswith('L'+str(i)):
                layer_ports.append(model_ports[key])
                
        # Instantiate wire to connect output to next layer's input
        if (i != (len(layers)-1)):
            neuron_count = layers[i].get_params()['NUM_NEURONS'].value * layers[i].get_params()['NUM_COL'].value
            last_out = model.Wire('out_'+str(i)+'_in_'+str(i+1), neuron_count)
            layer_ports.append(last_out)
        else:
            layer_ports.append(model_ports['model_output'])

    model.Instance(layer, 'L'+str(i)+'_'+layer.name, params = layer_params,
                  ports = layer_ports)

L0_NUM_COL
L0_NUM_NEURONS
L0_NUM_DEND
L0_P_DIST
L0_P_PROX
L0_NUM_SEG
L0_WRES_DIST
L0_WRES_PROX
L0_THRESHOLD
L1_NUM_COL
L1_NUM_NEURONS
L1_NUM_DEND
L1_P_DIST
L1_P_PROX
L1_NUM_SEG
L1_WRES_DIST
L1_WRES_PROX
L1_THRESHOLD
L2_NUM_COL
L2_NUM_NEURONS
L2_NUM_DEND
L2_P_DIST
L2_P_PROX
L2_NUM_SEG
L2_WRES_DIST
L2_WRES_PROX
L2_THRESHOLD


In [10]:
gen_file, rtl_path = gen_verilog(module = model, filename = 'model.v')

In [41]:
layer = myModel.layers[0]

In [44]:
params = layer.get_params()
param = params['NUM_COL']

In [46]:
param.name

'NUM_COL'

In [2]:
myModel = Model()

In [3]:
myModel.add(Layers.TNN_Layer(num_col=2, num_neurons=4, num_dend=2, p_dist=2, p_prox=1, num_seg=3, wres_dist=3, wres_prox=3, thres=6))

In [4]:
myModel.add(Layers.TNN_Layer(num_col=1, num_neurons=4, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=1, wres_prox=1, thres=6))

In [5]:
myModel.add(Layers.TNN_Layer(num_col=1, num_neurons=2, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=1, wres_prox=1, thres=6))

In [7]:
myModel.summary()

Layer 0
     NUM_COL 2
     NUM_NEURONS 4
     NUM_DEND 2
     P_DIST 2
     P_PROX 1
     NUM_SEG 3
     WRES_DIST 3
     WRES_PROX 3
     THRESHOLD 6
Layer 1
     NUM_COL 1
     NUM_NEURONS 4
     NUM_DEND 1
     P_DIST 2
     P_PROX 1
     NUM_SEG 1
     WRES_DIST 3
     WRES_PROX 3
     THRESHOLD 6


In [8]:
myModel.compile()

In [9]:
myModel.model

In [10]:
gen_file, rtl_path = gen_verilog(module = myModel.model, filename = 'model.v')